Generate 50m x 50m spatial grid layers for the top metropolitan areas in the US and Canada and save each grid as a .parquet file.

In [1]:
import os
import geopandas as gpd
from shapely.geometry import Polygon
import numpy as np
from tqdm import tqdm
from datetime import datetime
import gc # ?

In [2]:
# Set working directory to your base directory (update this path)
os.chdir("/Users/jpg23/data/downtownrecovery/commercial_districts/commercial_districts_paper/")

In [3]:
# Load the top 300 metropolitan areas file and convert it to EPSG:3857 projection
ma_america = gpd.read_parquet("top300metros.parquet")
ma_america = ma_america.to_crs('EPSG:3857')

In [4]:
ma_america.head()

,name,population,geometry
0,"New York-Newark-Jersey City, NY-NJ-PA",19908595.0,"MULTIPOLYGON (((-8019103.214 5049263.521, -801..."
1,"Los Angeles-Long Beach-Anaheim, CA",13111917.0,"MULTIPOLYGON (((-13202983.084 3958997.374, -13..."
2,"Chicago-Naperville-Elgin, IL-IN-WI",9566955.0,"MULTIPOLYGON (((-9900994.403 5170728.605, -990..."
3,"Dallas-Fort Worth-Arlington, TX",7673379.0,"MULTIPOLYGON (((-10916612.1 3841671.927, -1091..."
4,"Houston-The Woodlands-Sugar Land, TX",7142603.0,"MULTIPOLYGON (((-10543992.48 3468744.079, -105..."


In [5]:
# already_done = [
#     "New York-Newark-Jersey City, NY-NJ-PA",
#     "Los Angeles-Long Beach-Anaheim, CA",
#     "Chicago-Naperville-Elgin, IL-IN-WI",
#     "Dallas-Fort Worth-Arlington, TX",
#     "Houston-The Woodlands-Sugar Land, TX",
#     "Washington-Arlington-Alexandria, DC-VA-MD-WV",
#     "Philadelphia-Camden-Wilmington, PA-NJ-DE-MD",
#     "Toronto, ON",
#     "Miami-Fort Lauderdale-Pompano Beach, FL",
#     "Atlanta-Sandy Springs-Alpharetta, GA",
#     "Boston-Cambridge-Newton, MA-NH",
#     "Phoenix-Mesa-Chandler, AZ",
#     "San Francisco-Oakland-Berkeley, CA",
#     "Riverside-San Bernardino-Ontario, CA",
#     "Detroit-Warren-Dearborn, MI",
#     "Montreal, QC",
#     "Seattle-Tacoma-Bellevue, WA",
#     "Minneapolis-St. Paul-Bloomington, MN-WI",
#     "San Diego-Chula Vista-Carlsbad, CA",
#     "Tampa-St. Petersburg-Clearwater, FL",
#     "Denver-Aurora-Lakewood, CO",
#     "Baltimore-Columbia-Towson, MD",
#     "St. Louis, MO-IL",
#     "Orlando-Kissimmee-Sanford, FL",
#     "Charlotte-Concord-Gastonia, NC-SC",
#     "Vancouver, BC",
#     "San Antonio-New Braunfels, TX",
#     "Portland-Vancouver-Hillsboro, OR-WA",
#     "Sacramento-Roseville-Folsom, CA",
#     "Pittsburgh, PA",
#     "Austin-Round Rock-Georgetown, TX",
#     "Las Vegas-Henderson-Paradise, NV",
#     "Cincinnati, OH-KY-IN",
#     "Kansas City, MO-KS",
#     "Columbus, OH",
#     "Indianapolis-Carmel-Anderson, IN",
#     "Cleveland-Elyria, OH",
#     "Nashville-Davidson--Murfreesboro--Franklin, TN",
#     "San Jose-Sunnyvale-Santa Clara, CA",
#     "Virginia Beach-Norfolk-Newport News, VA-NC",
#     "Providence-Warwick, RI-MA",
#     "Jacksonville, FL",
#     "Milwaukee-Waukesha, WI",
#     "Ottawa - Gatineau, ON-QC",
#     "Calgary, AB",
#     "Oklahoma City, OK",
#     "Raleigh-Cary, NC",
#     "Edmonton, AB",
#     "Memphis, TN-MS-AR",
#     "Richmond, VA",
#     "Louisville, KY-IN",
#     "New Orleans-Metairie, LA",
#     "Salt Lake City, UT",
#     "Hartford-East Hartford-Middletown, CT",
#     "Buffalo-Cheektowaga, NY",
#     "Birmingham-Hoover, AL",
#     "Grand Rapids-Kentwood, MI",
#     "Rochester, NY",
#     "Tucson, AZ",
#     "Tulsa, OK",
#     "Urban Honolulu, HI",
#     "Fresno, CA",
#     "Worcester, MA-CT",
#     "Omaha-Council Bluffs, NE-IA",
#     "Bridgeport-Stamford-Norwalk, CT",
#     "Greenville-Anderson, SC",
#     "Albuquerque, NM",
#     "Bakersfield, CA",
#     "Albany-Schenectady-Troy, NY",
#     "Knoxville, TN",
#     "McAllen-Edinburg-Mission, TX",
#     "Baton Rouge, LA",
#     "El Paso, TX",
#     "New Haven-Milford, CT",
#     "Allentown-Bethlehem-Easton, PA-NJ",
#     "North Port-Sarasota-Bradenton, FL",
#     "Oxnard-Thousand Oaks-Ventura, CA",
#     "Quebec, QC",
#     "Winnipeg, MB",
#     "Columbia, SC",
#     "Dayton-Kettering, OH",
#     "Charleston-North Charleston, SC",
#     "Hamilton, ON",
#     "Stockton, CA",
#     "Greensboro-High Point, NC",
#     "Cape Coral-Fort Myers, FL",
#     "Boise City, ID",
#     "Colorado Springs, CO",
#     "Little Rock-North Little Rock-Conway, AR",
#     "Lakeland-Winter Haven, FL",
#     "Des Moines-West Des Moines, IA",
#     "Akron, OH",
#     "Poughkeepsie-Newburgh-Middletown, NY",
#     "Ogden-Clearfield, UT",
#     "Springfield, MA",
#     "Madison, WI",
#     "Winston-Salem, NC",
#     "Provo-Orem, UT",
#     "Deltona-Daytona Beach-Ormond Beach, FL",
#     "Syracuse, NY",
#     "Durham-Chapel Hill, NC",
#     "Wichita, KS",
#     "Toledo, OH",
#     "Augusta-Richmond County, GA-SC",
#     "Palm Bay-Melbourne-Titusville, FL",
#     "Harrisburg-Carlisle, PA",
#     "Jackson, MS",
#     "Spokane-Spokane Valley, WA",
#     "Kitchener - Cambridge - Waterloo, ON",
#     "Scranton--Wilkes-Barre, PA",
#     "Chattanooga, TN-GA",
#     "Lancaster, PA",
#     "Portland-South Portland, ME",
#     "Modesto, CA",
#     "Fayetteville-Springdale-Rogers, AR",
#     "London, ON",
#     "Youngstown-Warren-Boardman, OH-PA",
#     "Lansing-East Lansing, MI",
#     "Fayetteville, NC",
#     "Lexington-Fayette, KY",
#     "Anchorage, AK",
#     "Victoria, BC"
#     "Beaumont-Port Arthur, TX"
#     "Shreveport-Bossier City, LA"
#     "Tallahassee, FL"
#     "Montgomery, AL"
#     "Trenton-Princeton, NJ"
#     "Davenport-Moline-Rock Island, IA-IL"
#     "Eugene-Springfield, OR"
#     "Naples-Marco Island, FL"
#     "Ocala, FL"
#     "Ann Arbor, MI"
#     "Hickory-Lenoir-Morganton, NC"
#     "Fort Collins, CO"
#     "Huntington-Ashland, WV-KY-OH"
#     "Gainesville, FL"
#     "Lincoln, NE"
#     "Rockford, IL"
#     "Greeley, CO"
#     "Spartanburg, SC"
#     "Boulder, CO"
#     "Green Bay, WI"
#     "Columbus, GA-AL"
#     "South Bend-Mishawaka, IN-MI"
#     "Clarksville, TN-KY"
#     "Lubbock, TX"
#     "Saskatoon, SK"
#     "Roanoke, VA"
#     "Evansville, IN-KY"
#     "Kingsport-Bristol, TN-VA"
#     "Kennewick-Richland, WA"
#     "Hagerstown-Martinsburg, MD-WV"
#     "Olympia-Lacey-Tumwater, WA"
#     "Duluth, MN-WI"
#     "Utica-Rome, NY"
#     "Wilmington, NC"
#     "Crestview-Fort Walton Beach-Destin, FL"
#     "Longview, TX"
#     "Merced, CA"
#     "San Luis Obispo-Paso Robles, CA"
#     "Waco, TX"
#     "Sioux Falls, SD"
#     "Cedar Rapids, IA"
#     "Bremerton-Silverdale-Port Orchard, WA"
#     "Atlantic City-Hammonton, NJ"
#     "Tuscaloosa, AL"
#     "Erie, PA"
#     "College Station-Bryan, TX"
#     "Amarillo, TX"
#     "Santa Cruz-Watsonville, CA"
#     "Norwich-New London, CT"
#     "Laredo, TX"
#     "Lynchburg, VA"
#     "Kalamazoo-Portage, MI"
#     "Charleston, WV"
#     "Yakima, WA"
#     "Fargo, ND-MN"
#     "Regina, SK"
#     "Binghamton, NY"
#     "Fort Smith, AR-OK"
#     "Appleton, WI"
#     "Prescott Valley-Prescott, AZ"
#     "Tyler, TX"
#     "Daphne-Fairhope-Foley, AL"
#     "Macon-Bibb County, GA"
#     "Topeka, KS"
#     "Barnstable Town, MA"
#     "Sherbrooke, QC"
#     "Bellingham, WA"
#     "Rochester, MN"
#     "Burlington-South Burlington, VT"
#     "Lafayette-West Lafayette, IN"
#     "Champaign-Urbana, IL"
#     "Medford, OR"
#     "Kelowna, BC"
#     "Charlottesville, VA"
#     "Lebanon, NH-VT Micro Area"
#     "Las Cruces, NM"
#     "Hilton Head Island-Bluffton, SC"
#     "Lake Charles, LA"
#     "Athens-Clarke County, GA"
#     "Lake Havasu City-Kingman, AZ"
#     "Chico, CA"
#     "Barrie, ON"
#     "St. John's, NL"
#     "Columbia, MO"
#     "Springfield, IL"
#     "Johnson City, TN"
#     "Elkhart-Goshen, IN"
#     "Houma-Thibodaux, LA"
#     "Monroe, LA"
#     "Gainesville, GA"
#     "Yuma, AZ"
#     "Jacksonville, NC"
#     "Hilo, HI Micro Area"
#     "Florence, SC"
#     "St. Cloud, MN"
# ]

# print(ma_america.shape[0])
# ma_america = ma_america[~ma_america['name'].isin(already_done)]
# print(ma_america.shape[0])

In [6]:
# last_2 = ["Anchorage, AK", "Urban Honolulu, HI"]
# ma_america = ma_america[ma_america['name'].isin(last_2)]
ma_america = ma_america[ma_america['name']=='Anchorage, AK']
ma_america

,name,population,geometry
152,"Anchorage, AK",399335.0,"MULTIPOLYGON (((-16729745.297 8655051.071, -16..."


In [7]:
# Loop through each metropolitan area in the dataset
for x in ma_america.index:
    # Extract a single metropolitan area boundary
    temp_ma = ma_america.loc[ma_america.index == x].copy()
    nm = temp_ma['name'].tolist()[0]  # Get the metro area name
    # print(nm)
    
    # Define the output filename for the grid
    out_grid_nm = './MA_grid_bound/base_grid_' + nm + '.parquet'
    # print(out_grid_nm)
    
    # If the grid for this metro area already exists, skip it
    if os.path.isfile(out_grid_nm):
        pass
    else:
        print(nm)  # Print city name being processed
        start = datetime.now()  # Start timing the process

        # Set grid cell size (50m x 50m)
        width = 50
        height = 50
    
        # Get bounding box coordinates for the metro area
        xmin, ymin, xmax, ymax = temp_ma.total_bounds

        # Calculate how many rows and columns are needed to cover the bounding box
        rows = int(np.ceil((ymax - ymin) / height))
        cols = int(np.ceil((xmax - xmin) / width))

        # Initialize origin points for grid generation
        XleftOrigin = xmin
        XrightOrigin = xmin + width
        YtopOrigin = ymax
        YbottomOrigin = ymax - height

        polygons = []  # List to hold each grid cell polygon

        # Generate grid cells by looping through columns and rows
        for i in tqdm(range(cols)):  # Iterate over columns
            Ytop = YtopOrigin
            Ybottom = YbottomOrigin
            for j in range(rows):  # Iterate over rows
                # Create a rectangular polygon for each grid cell
                polygons.append(
                    Polygon([
                        (XleftOrigin, Ytop),
                        (XrightOrigin, Ytop),
                        (XrightOrigin, Ybottom),
                        (XleftOrigin, Ybottom)
                    ])
                )
                # Move down to the next row
                Ytop -= height
                Ybottom -= height

            # Move to the next column
            XleftOrigin += width
            XrightOrigin += width
            
        # Create a GeoDataFrame from the generated polygons
        grid = gpd.GeoDataFrame({'geometry': polygons})
        grid = grid.reset_index(drop=False)  # Add index as ID
        grid.columns = ['id', 'geometry']  # Name the columns
        grid.crs = 'EPSG:3857'  # Set projection
        
        # Save the grid to a parquet file
        out_grid_nm = './MA_grid_bound/base_grid_' + nm + '.parquet'
        grid.to_parquet(out_grid_nm)
        
        # Print how long it took to process
        end = datetime.now()
        print('Duration: {}'.format(end - start))

        # After saving the grid to parquet - delete data for that region
        del grid, polygons, temp_ma
        gc.collect()

Anchorage, AK


100%|██████████| 14645/14645 [33:15<00:00,  7.34it/s]  


Duration: 0:59:05.444637


Are Anchorage & Honolulu already in Byeonghwa's dataset?

In [8]:
# existing = gpd.read_file('/Users/jpg23/data/downtownrecovery/commercial_districts/byeonghwa_commercial_districts.geojson')
# existing.head()

In [9]:
# these2 = existing[existing['MSA_NAME'].isin(['Urban Honolulu', 'Anchorage'])]
# these2['MSA_NAME'].unique()

In [10]:
# existing[existing['STATE/PROVINCE']=='AK']

Honolulu is; Anchorage is not.